In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter
import math


In [2]:
src_sentences = [
    ["hello", "world", "."],
    ["how", "are", "you", "?"],
    ["i", "am", "fine", "."],
    ["this", "is", "a", "test"],
    ["another", "example"]
]
trg_sentences = [
    ["hello", "world", "!"],
    ["how", "do", "you", "do", "?"],
    ["i", "am", "doing", "well", "."],
    ["this", "is", "a", "reference"],
    ["one", "more", "example"]
]

dataset = list(zip(src_sentences, trg_sentences))


In [3]:
def build_vocab(sentences, pad=0, sos=1, eos=2):
    words = set(word for sent in sentences for word in sent)
    vocab = {word: idx+3 for idx, word in enumerate(words)}
    vocab['<pad>'] = pad
    vocab['<sos>'] = sos
    vocab['<eos>'] = eos
    itos = {idx: word for word, idx in vocab.items()}
    return vocab, itos

SRC_vocab, SRC_itos = build_vocab(src_sentences)
TRG_vocab, TRG_itos = build_vocab(trg_sentences)

In [4]:
def tokens_to_words(tokens, itos):
    return [itos[t] for t in tokens if t in itos]

In [5]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, batch_first=True)
    def forward(self, src):
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.rnn(embedded)
        return outputs, hidden, cell

In [7]:
class Attention(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()
        self.hid_dim = hid_dim
    def forward(self, hidden, encoder_outputs):
        hidden = hidden.squeeze(0)
        scores = torch.bmm(encoder_outputs, hidden.unsqueeze(2)).squeeze(2)
        attn_weights = F.softmax(scores, dim=1)
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs).squeeze(1)
        return context, attn_weights

In [8]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, attention):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim + hid_dim, hid_dim, batch_first=True)
        self.fc_out = nn.Linear(hid_dim * 2, output_dim)
        self.attention = attention
    def forward(self, input, hidden, cell, encoder_outputs):
        embedded = self.embedding(input.unsqueeze(1))
        context, attn_weights = self.attention(hidden, encoder_outputs)
        rnn_input = torch.cat([embedded, context.unsqueeze(1)], dim=2)
        output, (hidden, cell) = self.rnn(rnn_input, (hidden, cell))
        output = torch.cat([output.squeeze(1), context], dim=1)
        prediction = self.fc_out(output)
        return prediction, hidden, cell, attn_weights

In [9]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

In [10]:
INPUT_DIM = len(SRC_vocab)
OUTPUT_DIM = len(TRG_vocab)
EMB_DIM = 32
HID_DIM = 64

attn = Attention(HID_DIM)
enc = Encoder(INPUT_DIM, EMB_DIM, HID_DIM)
dec = Decoder(OUTPUT_DIM, EMB_DIM, HID_DIM, attn)
model = Seq2Seq(enc, dec)

In [11]:
def greedy_decode(model, src, max_len=10, sos_idx=1, eos_idx=2):
    src_tensor = torch.tensor([[SRC_vocab[w] for w in src]])
    encoder_outputs, hidden, cell = model.encoder(src_tensor)
    input_token = torch.tensor([sos_idx])
    result = [sos_idx]
    for _ in range(max_len):
        output, hidden, cell, _ = model.decoder(input_token, hidden, cell, encoder_outputs)
        next_token = output.argmax(1).item()
        result.append(next_token)
        if next_token == eos_idx:
            break
        input_token = torch.tensor([next_token])
    return result

In [12]:
def beam_search(model, src, beam_width=3, max_len=10, sos_idx=1, eos_idx=2):
    src_tensor = torch.tensor([[SRC_vocab[w] for w in src]])
    encoder_outputs, hidden, cell = model.encoder(src_tensor)
    beam = [([sos_idx], 0.0, hidden, cell)]
    completed = []
    for _ in range(max_len):
        new_beam = []
        for seq, score, hidden, cell in beam:
            if seq[-1] == eos_idx:
                completed.append((seq, score))
                continue
            input_token = torch.tensor([seq[-1]])
            output, h, c, _ = model.decoder(input_token, hidden, cell, encoder_outputs)
            log_probs = F.log_softmax(output, dim=1)
            topk = torch.topk(log_probs, beam_width)
            for i in range(beam_width):
                next_token = topk.indices[0][i].item()
                next_score = score + topk.values[0][i].item()
                new_seq = seq + [next_token]
                new_beam.append((new_seq, next_score, h, c))
        beam = sorted(new_beam, key=lambda x: x[1], reverse=True)[:beam_width]
    if completed:
        best_seq = sorted(completed, key=lambda x: x[1], reverse=True)[0][0]
    else:
        best_seq = beam[0][0]
    return best_seq

In [13]:
def get_ngrams(tokens, n):
    return Counter([tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)])
def clipped_precision(hyp, ref, n):
    hyp_ngrams = get_ngrams(hyp, n)
    ref_ngrams = get_ngrams(ref, n)
    clipped = sum(min(hyp_ngrams[g], ref_ngrams.get(g,0)) for g in hyp_ngrams)
    total = sum(hyp_ngrams.values())
    return clipped / total if total > 0 else 0
def brevity_penalty(hyp, ref):
    c,r = len(hyp), len(ref)
    return 1 if c>r else math.exp(1 - r/c)
def bleu_score(hyp, ref):
    log_prec = []
    for i in range(1,5):
        p = clipped_precision(hyp, ref, i)
        if p==0: return 0
        log_prec.append(0.25 * math.log(p))
    geo_mean = math.exp(sum(log_prec))
    bp = brevity_penalty(hyp, ref)
    return bp * geo_mean

In [14]:
for i in range(len(dataset)):
    src, ref = dataset[i]
    greedy = greedy_decode(model, src)
    beam = beam_search(model, src, beam_width=5)

    print("SOURCE:   ", src)
    print("GREEDY:   ", tokens_to_words(greedy, TRG_itos))
    print("BEAM:     ", tokens_to_words(beam, TRG_itos))
    print("REFERENCE:", ref)
    print("-"*50)


SOURCE:    ['hello', 'world', '.']
GREEDY:    ['<sos>', 'example', 'i', 'i', 'i', 'i', 'i', 'i', 'i', 'i', 'i']
BEAM:      ['<sos>', 'how', 'how', 'how', '<eos>']
REFERENCE: ['hello', 'world', '!']
--------------------------------------------------
SOURCE:    ['how', 'are', 'you', '?']
GREEDY:    ['<sos>', 'how', 'how', 'how', 'how', 'how', 'how', 'how', 'how', 'how', 'how']
BEAM:      ['<sos>', 'how', 'how', '<eos>']
REFERENCE: ['how', 'do', 'you', 'do', '?']
--------------------------------------------------
SOURCE:    ['i', 'am', 'fine', '.']
GREEDY:    ['<sos>', '?', '?', 'more', 'this', 'more', 'example', 'this', 'well', 'reference', 'this']
BEAM:      ['<sos>', '?', 'how', 'how', 'how', '<eos>']
REFERENCE: ['i', 'am', 'doing', 'well', '.']
--------------------------------------------------
SOURCE:    ['this', 'is', 'a', 'test']
GREEDY:    ['<sos>', 'how', 'how', 'how', 'how', 'how', 'how', 'how', 'how', 'how', 'how']
BEAM:      ['<sos>', 'how', 'how', '<eos>']
REFERENCE: ['this',